# M-You gợi ý cho một mục tiêu (`POST /goal-suggestions`)

Nhập **CIF** và **tên mục tiêu** ở ô đầu tiên rồi chạy lần lượt hai ô. Agent tự đọc toàn bộ dữ liệu của khách từ `pfm` (mục tiêu, tài sản, giao dịch 3 tháng, hũ, tài khoản), để model **gemma** viết insight, rồi hiển thị đúng dạng thẻ trên màn Mục tiêu: ô trên (điều M-You nhận thấy) và ô có màu (Gợi ý cho bạn, kèm chữ "Chi tiết"). Thẻ chỉ có hai phần, không chia loại; độ dài tối đa 320 và 140 ký tự.

Cần mạng và `AI_PLATFORM_API_KEY` trong `../.env`. Model mặc định `google/gemma-4-31b-it`, đổi bằng biến `GOAL_SUGGEST_MODEL`. Mỗi lần chạy mất khoảng 3 đến 15 giây.

Đặt `SHOW_DATA = True` để xem nguyên văn dữ liệu mà model đã đọc, gồm các phương án điều chỉnh và các điểm đáng chú ý được code tính sẵn.

In [ ]:
CIF = "CIF_0001"
GOAL_NAME = "Mua Xe VF3"   # tên mục tiêu (không phân biệt hoa thường / dấu); hoặc đặt GOAL_ID = "goal-13"
GOAL_ID = None
SHOW_DATA = False

In [ ]:
import os
import sys
from html import escape
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import HTML, display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
load_dotenv(REPO_ROOT / ".env")
assert os.environ.get("AI_PLATFORM_API_KEY"), f"Set AI_PLATFORM_API_KEY in {REPO_ROOT / '.env'} first"

from goal_suggest import data, digest
from goal_suggest.contracts import GoalSuggestionRequest
from goal_suggest.service import DEFAULT_MODEL, suggest
from tools.goal_tools import summarize_goals

def card(insight) -> str:
    """The goal-screen card: the top box, then the coloured box with the "Chi tiết" link."""
    return f"""
<div style="max-width:380px;font-family:system-ui,sans-serif;background:#fff;border-radius:16px;padding:16px;box-shadow:0 2px 10px #0002">
  <div style="display:flex;align-items:center;gap:10px;margin-bottom:10px">
    <div style="width:32px;height:32px;border-radius:50%;background:#ff6a13;color:#fff;display:flex;align-items:center;justify-content:center;font-weight:700">M</div>
    <b style="font-size:16px">M-You gợi ý</b>
  </div>
  <p style="font-size:15px;line-height:24px;margin:0 0 12px">{escape(insight.observation)}</p>
  <div style="background:#fff1ea;border-radius:16px;padding:12px 14px">
    <div style="font-size:13px;font-weight:600;color:#c2410c;margin-bottom:4px">Gợi ý cho bạn</div>
    <div style="font-size:14px;line-height:20px">{escape(insight.suggestion)}
      <span style="color:#c2410c;font-weight:600;white-space:nowrap">Chi tiết ›</span></div>
  </div>
</div>"""

In [ ]:
# Chạy ô này để tạo insight cho (CIF, mục tiêu) đã nhập ở ô đầu tiên.
raw = data.collect(CIF)
summary = summarize_goals(raw["snapshot"])
print(f"Model: {os.environ.get('GOAL_SUGGEST_MODEL') or DEFAULT_MODEL} | hôm nay (đồng hồ pfm): {summary['today']}")
print("Mục tiêu của khách:", [(g["id"], g["name"], g["status"]) for g in summary["goals"]])

try:
    goal, text = digest.build(raw, GOAL_ID, GOAL_NAME)
except digest.GoalNotFound:
    raise SystemExit("Không tìm thấy mục tiêu này: chọn một tên (hoặc mã) trong danh sách trên.")
except digest.GoalAmbiguous as exc:
    raise SystemExit(f"Có nhiều mục tiêu trùng tên ({exc}): đặt GOAL_ID thay cho GOAL_NAME.")

if SHOW_DATA:
    print("\n----- DỮ LIỆU MODEL ĐỌC -----\n" + text + "\n-----------------------------")

resp = await suggest(GoalSuggestionRequest(user_id=CIF, goal_id=goal["id"]))
if resp.insight is None:
    print(f"Chưa có insight (model lỗi hoặc lời bị từ chối): {resp.reason}")
else:
    i = resp.insight
    print(f"Mục tiêu: {i.goal_name} ({i.goal_id}) | ô trên {len(i.observation)} ký tự, ô gợi ý {len(i.suggestion)} ký tự")
    print(f"Câu hỏi khi bấm Chi tiết ({len(i.question)} ký tự): {i.question}")
    display(HTML(card(i)))